# 08 空間流病 — 參考解答

松柏護理之家退伍軍人症群聚事件空間分析練習的完整解答。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
df["died"] = (df["outcome"] == "died").astype(int)

## 題目 1：致死率空間分布

In [ ]:
# 計算 floor × wing 致死率
spatial = df.groupby(["floor", "wing"]).agg(
    total=("case_id", "count"),
    infected=("infected", "sum"),
    died=("died", "sum"),
).reset_index()
spatial["attack_rate"] = (spatial["infected"] / spatial["total"] * 100).round(1)
spatial["cfr"] = (spatial["died"] / spatial["infected"] * 100).round(1)

print("=== 翼區侵襲率 & 致死率 ===")
print(spatial[["floor", "wing", "total", "infected", "died", "attack_rate", "cfr"]].to_string(index=False))

# 致死率熱力圖
heatmap_cfr = spatial.pivot(index="floor", columns="wing", values="cfr")

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(heatmap_cfr, annot=True, fmt=".1f", cmap="Reds",
            cbar_kws={"label": "%"}, ax=ax)
ax.set_title("致死率 (%) by Floor \u00d7 Wing")
ax.set_ylabel("Floor")
plt.tight_layout()
plt.show()

# 解讀
highest_cfr = spatial.loc[spatial["cfr"].idxmax()]
highest_ar = spatial.loc[spatial["attack_rate"].idxmax()]
print(f"\n致死率最高：{highest_cfr['floor']}F-{highest_cfr['wing']}（{highest_cfr['cfr']}%）")
print(f"侵襲率最高：{highest_ar['floor']}F-{highest_ar['wing']}（{highest_ar['attack_rate']}%）")
print("\n\u2192 致死率最高的翼區不一定是侵襲率最高的翼區")
print("\u2192 致死率還受住民特性（年齡、共病）影響，不完全取決於暴露強度")

## 題目 2：淋浴使用的空間分布

In [ ]:
# 淋浴使用比例
shower = df.groupby(["floor", "wing"]).agg(
    total=("case_id", "count"),
    shower_users=("shower_use", "sum"),
    infected=("infected", "sum"),
).reset_index()
shower["shower_pct"] = (shower["shower_users"] / shower["total"] * 100).round(1)
shower["attack_rate"] = (shower["infected"] / shower["total"] * 100).round(1)

print("=== 淋浴比例 vs 侵襲率 ===")
print(shower[["floor", "wing", "shower_pct", "attack_rate"]].to_string(index=False))

# 並排熱力圖
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

hm_shower = shower.pivot(index="floor", columns="wing", values="shower_pct")
sns.heatmap(hm_shower, annot=True, fmt=".1f", cmap="Blues",
            cbar_kws={"label": "%"}, ax=axes[0])
axes[0].set_title("淋浴使用比例 (%)")
axes[0].set_ylabel("Floor")

hm_ar = shower.pivot(index="floor", columns="wing", values="attack_rate")
sns.heatmap(hm_ar, annot=True, fmt=".1f", cmap="YlOrRd",
            cbar_kws={"label": "%"}, ax=axes[1])
axes[1].set_title("侵襲率 (%)")
axes[1].set_ylabel("Floor")

plt.tight_layout()
plt.show()

# 相關性
corr = shower[["shower_pct", "attack_rate"]].corr().iloc[0, 1]
print(f"\n淋浴比例 vs 侵襲率相關係數：r = {corr:.3f}")
print("\n\u2192 如果兩張熱力圖的高低分布類似，支持水源傳播假說")
print("\u2192 但也要考慮交絡因子（如 functional_status 影響淋浴能力，Ch05 已分析）")

## 題目 3（挑戰題）：高風險房間清單

In [ ]:
# 每間房侵襲率
room_stats = df.groupby("room").agg(
    total=("case_id", "count"),
    infected=("infected", "sum"),
).reset_index()
room_stats["attack_rate"] = (room_stats["infected"] / room_stats["total"] * 100).round(1)

# 解析 floor 和 wing
room_stats["floor"] = room_stats["room"].str[0].astype(int)
room_stats["wing"] = room_stats["room"].str[1]

# 篩選 >= 75%
high_risk = (
    room_stats[room_stats["attack_rate"] >= 75]
    .sort_values("attack_rate", ascending=False)
    [["room", "total", "infected", "attack_rate", "floor", "wing"]]
)

print(f"=== 高風險房間清單（侵襲率 \u2265 75%）===")
print(f"共 {len(high_risk)} 間\n")
print(high_risk.to_string(index=False))

# 按翼區統計
print("\n=== 高風險房間的翼區分布 ===")
wing_counts = high_risk.groupby(["floor", "wing"]).size().reset_index(name="high_risk_rooms")
print(wing_counts.to_string(index=False))

print("\n\u2192 提交此清單給感控團隊，優先對這些房間進行環境採檢")
print("\u2192 特別關注高風險房間集中的翼區，檢查蓮蓬頭和熱水管線")

### 解讀

- **致死率 vs 侵襲率**：兩者不一定正相關。侵襲率反映暴露風險，致死率反映宿主脆弱度
- **淋浴 × 空間**：如果淋浴比例高的翼區也是侵襲率高的翼區，空間分析強化了水源傳播假說
- **高風險房間**：集中在特定翼區的高風險房間，提示該翼區的供水系統可能是傳播途徑
- **行動建議**：對高風險翼區的蓮蓬頭、熱水管線進行退伍軍人菌培養與環境採檢